# Optimal policy estimation with MonteCarlo

Consider the MDP carachterized by $(\mathcal{S}, \mathcal{A}, \mathcal{P}, \mathcal{R}, \gamma)$.

We consider the common case where the probabilities of transition are unknown, i.e. $\mathcal{P}:=\{P^{a}:\, a\in\mathcal{A}\}$ is unknown.

>A common approach to approximate the optimal policy consists in alternating the following two steps:
>1) estimating the value function by MonteCarlo;
>2) updating the policy by chosing the action that maximizes the value function.

**The MonteCarlo estimation**:

Under the assumptions of a stationary policy, consider the value function

$$
v_{\pi}(s):=\mathbb{E}_{\pi} \left[ G_{0} \mid S_{0}=s \right],
$$

where $G_{0}:=\sum\limits_{k=0}^{\min\{T,\tau\}} \gamma^{k} R_{1+k} $ is the *objective function* at time $0$, with $R_{1+k}$ the reward at time $1+k$, $T$ a finite horizon and $\tau$ the minimum time to reach an absorbing state. The objective function selects a random trajectory starting from state $s$ under the policy $\pi$.

Let $G_{1},\dots,G_{n}$ be a sample of i.i.d. $n$ objective functions distributed as $G_{0}$, then the MonteCarlo estimator of the value function $v_{\pi}$ at time $n\in\mathbb{N}^{+}$ is the empirical average:

$$
\hat{v}_{\pi,n}(s) := \frac{1}{n} \sum\limits_{i=1}^{n} G_{i}(s).
$$

In [106]:
import numpy as np

In [107]:
def objective_function(R, pi, P_trans_allow, s0, gamma, T, absorbing_state=None):
    '''
    Computes the objective function G(s)
    by selecting a random trajectory,
    until T steps or an absorbing state is reached.

    R[s, a] : matrix of rewards
    pi[s, a] : policy, i.e. probability of choosing action a being in state s
    P_trans_allow[a,s,s'] : matrices P_{s,s'}^{a} with 1 where the transition is eventually allowed
                          : transition rates are not known!!
    s0: initial state to compute G(s0)
    gamma: discount factor
    T : horizon / length of a trajectory
    absorbing_state: list of absorbing states known by the expert

    output : objective function G(s0) for a random trajectory starting from s0
    '''

    if absorbing_state is None:    # if no absorbing states are provided, set empty list
        absorbing_state = []

    G = 0                        # objective function counter
    t = 0                        # time step counter
    s = s0                       # initial state

    n_states = R.shape[0]                            # number of states
    n_actions = R.shape[1]                           # number of actions

    while s not in absorbing_state and t < T:

        # Randomly choose new action to apply to s, according to policy
        new_action = np.random.choice(n_actions, p=pi[s])

        # Update objective function
        G += (gamma ** t) * R[s, new_action]

        # Compute next state
        possible_next_states = np.flatnonzero(P_trans_allow[new_action, s] > 0)   # allowed next states s',
                                                                                  # if action new_action applied to s
        if possible_next_states.size == 0:              # check there is at least one allowed state
            raise ValueError(
                f"No allowed next state for state {s}, action {new_action}. "
                "Each row of P_trans_allow[a, s, :] must contain at least one non-zero entry."
            )
        
        s = np.random.choice(possible_next_states)

        # Update steps
        t += 1

    return G


In [108]:
def value_function_MC(R, pi, P_trans_allow, gamma, T, N, absorbing_state=None):
    '''
    Computes the value function v(s) for a given policy pi
    by the Monte Carlo method

    R[s, a] : matrix of rewards
    pi[s, a] : policy
    P_trans_allow[a,s,s'] : matrices P_{s,s'}^{a} with 1 where the transition is eventually allowed
                          : transition rates are not known!!
    gamma: discount factor
    T : horizon / length of a trajectory
    N : number of trajectories
    absorbing_states: list of absorbing states known by the expert

    output: value function v_pi(s) estimated by Monte Carlo method
    '''

    # Save N rewards from N possible realisations, i.e. G_1, G_2, ..., G_N
    n_states = R.shape[0]                 # number of states
    rewards = np.zeros(n_states)          # store rewards for each state

    for s in range(n_states):             # generate N trajectories from each state s
        for _ in range(N):
            
            rewards[s] += objective_function(R, pi, P_trans_allow, s, gamma, T, absorbing_state)
  
    # Monte Carlo estimation
    return rewards / N


**A remark on the code:**

- For a state $s$, the action to apply $a$, is chosen under the policy $\pi(a\mid s)$.

- For a state $s$ and action $a$, the new state $s'$ is randomly unformly chosen from the matrix of allowed transitions, under the condition *P_trans_allowed $[a,s,s'] > 0 $.*

- This model assumes not known the transition probabilities to choose a state $s'$ starting from a state $s$ and an action $a$. Anyways, it takes into account some information about the environment: the way states are possibly connected under the choice of all actions.

## Example

Toy example from the course.

In [109]:
N = 1000        # number of trajectories
T = 100         # horizon / length of a trajectory

R_1 = np.array([[5, 1 ,0] ,[-1, 1 , 0]])                    # matrix of rewards

pi_policy = np.array([[1, 0, 0], [0, 0, 1]])                # policy to evaluate

P_s_allowed = np.array([[1, 1], [1, 1]])                    # environmental matrix P[a,s,s'] with 1 where the transition is eventually allowed
P_w_allowed = np.array([[1, 0], [0, 1]])
P_c_allowed = np.array([[1 ,0] ,[1 ,0]])
P_act_allowed = np.stack([P_s_allowed, P_w_allowed, P_c_allowed] , axis=0)

gamma_ex = 0.8

value_function_MC(R_1, pi_policy, P_act_allowed, gamma=gamma_ex, T=T, N=N)

array([17.90523751, 14.29922786])